# 06 — Imputation-aware model (v4)

**Bug found in 05:** LGBM trusts imputed POA (median 6.57) as if real.
At midday with missing POA, it predicts ~500 kW when actual is 0-680.

**Fix:** add `poa_is_missing`, `temp_is_missing`, `wind_is_missing` binary flags
BEFORE imputation. Retrain. Compare v3 vs v4 on the same test set.

## Setup

In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor

import solarops.pipeline
from solarops.pipeline import (
    load_raw, align_15m, split_time, impute_features, evaluate,
    add_time_features, FEATURES, TARGET_COL, IRRADIANCE_COL,
)

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data/raw/pvdaq/2023-solar-data-prize/system-2107"

raw = load_raw(DATA_DIR)
df  = align_15m(raw)
train, test = split_time(df, "2022-07-01")

## Step 1 — add missing flags BEFORE imputation

The flag must survive imputation (it's 0/1, never NaN, so the imputer leaves it alone).

In [3]:
MISSING_FLAGS = [f"{c}_is_missing" for c in FEATURES]

for df_ in (train, test):
    for feat, flag in zip(FEATURES, MISSING_FLAGS):
        df_[flag] = df_[feat].isna().astype(int)

print("Flag prevalence in test:")
print(test[MISSING_FLAGS].mean().round(3))

Flag prevalence in test:
poa_irradiance_o_149574_is_missing         0.027
ambient_temperature_o_149575_is_missing    0.001
wind_speed_o_149576_is_missing             0.000
dtype: float64


## Step 2 — impute the original features, keep the flags

We still impute (median), but now the tree sees **both** the imputed value and the "this was fake" signal.

In [4]:
X_train, X_test, y_train, y_test = impute_features(train, test)

train_fe = add_time_features(train)
test_fe  = add_time_features(test)
TIME_FEATURES = ["hour", "month", "dayofyear"]

# NO reset_index — X_train, train_fe, train[flags] all share train.index
X_train_v4 = pd.concat([X_train, train_fe[TIME_FEATURES], train[MISSING_FLAGS]], axis=1)
X_test_v4  = pd.concat([X_test,  test_fe[TIME_FEATURES],  test[MISSING_FLAGS]],  axis=1)

X_train_v3 = pd.concat([X_train, train_fe[TIME_FEATURES]], axis=1)
X_test_v3  = pd.concat([X_test,  test_fe[TIME_FEATURES]],  axis=1)

# sanity check — catch this bug class forever
assert len(X_train_v4) == len(y_train), f"train mismatch: {len(X_train_v4)} vs {len(y_train)}"
assert len(X_test_v4)  == len(y_test),  f"test mismatch: {len(X_test_v4)} vs {len(y_test)}"
print("shapes OK:", X_train_v4.shape, X_test_v4.shape)

shapes OK: (160396, 9) (46744, 9)


## Step 3 — retrain v3 (baseline) vs v4 (with missing flags)

Same hyperparameters, same test set. Only difference = 3 extra binary features.

In [5]:
X_train_v3 = pd.concat([X_train, train_fe[TIME_FEATURES]], axis=1)
X_test_v3  = pd.concat([X_test,  test_fe[TIME_FEATURES]],  axis=1)

v3 = LGBMRegressor(random_state=42, n_estimators=300, learning_rate=0.05).fit(X_train_v3, y_train)
v4 = LGBMRegressor(random_state=42, n_estimators=300, learning_rate=0.05).fit(X_train_v4, y_train)

results = pd.DataFrame([
    evaluate(v3, X_test_v3, y_test, "v3 (no flags)"),
    evaluate(v4, X_test_v4, y_test, "v4 (+missing flags)"),
]).round(3)
results

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000688 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1047
[LightGBM] [Info] Number of data points in the train set: 160396, number of used features: 6
[LightGBM] [Info] Start training from score 153.684736
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001498 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1053
[LightGBM] [Info] Number of data points in the train set: 160396, number of used features: 9
[LightGBM] [Info] Start training from score 153.684736


,model,MAE,RMSE,R2
0,v3 (no flags),25.690,50.785,0.952
1,v4 (+missing flags),25.692,51.000,0.951


## Step 4 — did the top-20 worst residuals shrink?

Re-run the same "worst residuals" check as notebook 05. The two imputation-artifact rows
(2022-11-21 13:45, 2022-10-23 11:45) should now be dramatically better.

In [6]:
pred4 = v4.predict(X_test_v4)

worst = pd.DataFrame({
    "measured_on": test["measured_on"].values,
    "actual":      y_test,
    "predicted":   pred4,
    "residual":    y_test - pred4,
    "poa":         X_test[IRRADIANCE_COL].values,
})
worst["abs_resid"] = worst["residual"].abs()

print("Top 10 worst residuals AFTER v4 fix:")
print(worst.nlargest(10, "abs_resid")[["measured_on","actual","predicted","residual","poa"]].to_string(index=False))

Top 10 worst residuals AFTER v4 fix:
        measured_on   actual  predicted    residual        poa
2022-11-01 10:30:00   0.0000 549.147293 -549.147293 712.133333
2023-03-18 11:45:00   0.0000 533.239506 -533.239506 671.433333
2022-11-01 14:00:00  69.7600 597.872978 -528.112978 992.033333
2022-11-21 13:30:00  10.0288 521.659919 -511.631119 754.100000
2022-11-21 13:45:00   9.7664 519.377418 -509.611018   6.566667
2023-03-18 11:30:00   0.0000 508.757822 -508.757822 635.366667
2022-11-01 13:45:00   0.0000 488.283490 -488.283490 714.400000
2023-03-04 11:30:00 616.9600 137.276328  479.683672 152.766667
2023-03-18 12:00:00   0.0000 475.264541 -475.264541 629.900000
2022-10-25 13:15:00 669.4400 198.558931  470.881069   6.566667


## Conclusions from the flag experiment

Flags didn't help (v4 ≈ v3). Reason: **missing-POA-at-noon is genuinely ambiguous**
— training contains both 0 kW and 669 kW for the same input.

**Don't try to predict through bad data. Flag it.**

## Reliability rules (part of the pipeline, not the model)

Every prediction gets a `trust` label:
- "ok"                → features present, |residual| < 200 kW
- "anomaly"           → features present, |residual| ≥ 200 kW  → investigate
- "unreliable_input"  → POA or temp was imputed at hour in [6,20]  → don't act on number

In [7]:
def trust_label(poa_imputed, temp_imputed, hour, residual):
    if poa_imputed or temp_imputed:
        if 6 <= hour <= 20:
            return "unreliable_input"
    if abs(residual) >= 200:
        return "anomaly"
    return "ok"

report = pd.DataFrame({
    "measured_on": test["measured_on"].values,
    "actual":      y_test,
    "predicted":   v4.predict(X_test_v4),
    "poa_imputed": test["poa_irradiance_o_149574_is_missing"].values,
    "hour":        test["measured_on"].dt.hour.values,
})
report["residual"] = report["actual"] - report["predicted"]
report["trust"]    = report.apply(
    lambda r: trust_label(r["poa_imputed"], False, r["hour"], r["residual"]), axis=1
)
print(report["trust"].value_counts())

trust
ok                  45961
anomaly               446
unreliable_input      337
Name: count, dtype: int64


## How many predictions does each bucket contain?
`ok` should be ~95%. `unreliable_input` ~5%. `anomaly` = the rows ops actually cares about.

In [8]:
print(report.groupby("trust")["residual"].agg(["count", "mean", "std", lambda x: (x.abs() > 200).mean()]).round(2))
print("\nSample anomalies:")
print(report[report["trust"]=="anomaly"].head(10))

                  count   mean     std  <lambda_0>
trust                                             
anomaly             446 -20.82  269.91        1.00
ok                45961   4.89   40.78        0.00
unreliable_input    337  77.03  167.58        0.27

Sample anomalies:
             measured_on  actual   predicted  poa_imputed  hour    residual  \
260  2022-07-03 17:00:00  476.80  240.668803            0    17  236.131197   
327  2022-07-04 09:45:00  219.52  457.394096            0     9 -237.874096   
335  2022-07-04 11:45:00  261.76  475.198167            0    11 -213.438167   
429  2022-07-05 11:15:00  320.00  596.947803            0    11 -276.947803   
432  2022-07-05 12:00:00  366.08  603.139535            0    12 -237.059535   
436  2022-07-05 13:00:00  222.08  457.627230            0    13 -235.547230   
1676 2022-07-18 11:00:00  192.64  430.161229            0    11 -237.521229   
2727 2022-07-29 09:45:00  202.88  405.448700            0     9 -202.568700   
2728 2022-07-29

## What we learned

1. Imputation flags alone (v4) didn't improve MAE — missing-POA-at-noon is
   genuinely ambiguous (train has both 0 kW and 669 kW with same input).
2. **Accept v3 as our best model (MAE 25.69).** Don't tune to fix bad data.
3. **Real fix is operational, not statistical:** label each prediction
   as `ok / anomaly / unreliable_input`. Ops cares about which bucket,
   not just the number.

## Action
- Save v3 as production model (next notebook: 07_save_model).
- Carry `trust_label()` forward into pipeline.py.
- For missing-POA rows, work on **input validation** (satellite irradiance
  fallback? neighbor interpolation? — future work).